# 04. Baseball Savant 영상 다운로드
- 코랩 전용
- Statcast parquet에서 X구간(9타자 이내) play_id 추출
- 배치(200개)단위 다운로드 → zip 묶어서 드라이브 저장
- 중간에 끊겨도 progress.csv 기준으로 이어서 실행 가능

## 담당 시즌 설정
- **나 (프로플러스)**: `MY_SEASONS = [2021, 2022, 2023]`
- **팀원 (무료)**: `MY_SEASONS = [2024, 2025]` + `BATCH_SIZE = 100`

In [4]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os

# ══════════════════════════════════════════════════════
# ★ 여기만 수정 ★
# 나 (프로플러스): MY_SEASONS = [2021, 2022, 2023] / BATCH_SIZE = 200
# 팀원 (무료)   : MY_SEASONS = [2024, 2025]        / BATCH_SIZE = 100
MY_SEASONS   = [2021, 2022, 2023]
X_MAX_BATTER = 9
BATCH_SIZE   = 200
# ══════════════════════════════════════════════════════

# parquet 파일 ID (2021~2025 순서)
PARQUET_FILE_IDS = {
    2021: '1KT17izBcO92VXTIr0LyyWGl-0bKHW7sS',
    2022: '16ak1N09M0ptyq4xjR-mpCBnY-L1dQadE',
    2023: '1bX2n3hpptTBZszPyB5EY2Ov0lCI4nM8I',
    2024: '17-ZhSO18-vjkSFXOZtQKEVTYnPmGjMuY',
    2025: '1uW_apJ4IC63FvTvK6BkZYBaAKA8ge5_L',
}

# zip/progress/play_id는 내 MyDrive에 저장
MY_BASE      = '/content/drive/MyDrive/MLB_pitcher'
VIDEO_DIR    = os.path.join(MY_BASE, 'video_zips')
PROGRESS_DIR = os.path.join(MY_BASE, 'data')
PARQUET_DIR  = '/content/parquets'   # 코랩 임시 저장
TEMP_DIR     = '/content/tmp_videos'

SEASON_KEY   = '_'.join(map(str, MY_SEASONS))
PROGRESS_CSV = os.path.join(PROGRESS_DIR, f'progress_{SEASON_KEY}.csv')
PLAY_ID_CSV  = os.path.join(PROGRESS_DIR, f'play_ids_{SEASON_KEY}.csv')

os.makedirs(VIDEO_DIR,    exist_ok=True)
os.makedirs(PROGRESS_DIR, exist_ok=True)
os.makedirs(PARQUET_DIR,  exist_ok=True)
os.makedirs(TEMP_DIR,     exist_ok=True)

print(f'담당 시즌  : {MY_SEASONS}')
print(f'X구간 기준 : 최대 {X_MAX_BATTER}타자 이내 투구')
print(f'배치 크기  : {BATCH_SIZE}개')
print(f'zip 저장   : {VIDEO_DIR}')
print(f'체크포인트 : {PROGRESS_CSV}')
print('경로 설정 완료 ✓')

Mounted at /content/drive
담당 시즌  : [2021, 2022, 2023]
X구간 기준 : 최대 9타자 이내 투구
배치 크기  : 200개
zip 저장   : /content/drive/MyDrive/MLB_pitcher/video_zips
체크포인트 : /content/drive/MyDrive/MLB_pitcher/data/progress_2021_2022_2023.csv
경로 설정 완료 ✓


In [ ]:
!pip install -q requests beautifulsoup4 gdown
import requests
from bs4 import BeautifulSoup
import pandas as pd
import gdown
import re
import time
import zipfile
import shutil
from pathlib import Path

## 1. game_pk 추출 → Baseball Savant 크롤링 → play_id 수집

In [ ]:
GP_CRAWLED_CSV = os.path.join(PROGRESS_DIR, f'gp_crawled_{SEASON_KEY}.csv')

REQ_HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36'
}

def crawl_play_ids_from_game(game_pk):
    """Baseball Savant 경기 페이지에서 play_id 목록 크롤링"""
    url = f'https://baseballsavant.mlb.com/game?game_pk={game_pk}'
    try:
        res = requests.get(url, headers=REQ_HEADERS, timeout=20)
        if res.status_code != 200:
            return []
        soup = BeautifulSoup(res.text, 'html.parser')
        play_ids = []
        for a in soup.find_all('a', href=True):
            m = re.search(r'[?&]playId=([a-f0-9\-]{36})', a['href'])
            if m:
                play_ids.append(m.group(1))
        return list(set(play_ids))
    except Exception:
        return []

# ── parquet 다운로드 ──────────────────────────────────────────────────────────
for season in MY_SEASONS:
    parquet_path = os.path.join(PARQUET_DIR, f'statcast_{season}.parquet')
    if os.path.exists(parquet_path):
        print(f'[{season}] parquet 이미 존재 → 건너뜀')
        continue
    file_id = PARQUET_FILE_IDS[season]
    print(f'[{season}] parquet 다운로드 중...')
    gdown.download(f'https://drive.google.com/uc?id={file_id}', parquet_path, quiet=False)
    print(f'[{season}] 완료')

# ── 선발 game_pk 목록 추출 ───────────────────────────────────────────────────
starter_gp = []
for season in MY_SEASONS:
    parquet_path = os.path.join(PARQUET_DIR, f'statcast_{season}.parquet')
    if not os.path.exists(parquet_path):
        print(f'[{season}] parquet 없음 → 건너뜀')
        continue
    df = pd.read_parquet(parquet_path)
    df = df[df['game_type'] == 'R'].copy()

    # 선발 조건: at_bat_number 최솟값이 1인 pitcher
    ab_min = (
        df.groupby(['game_pk', 'pitcher'])['at_bat_number']
        .min().reset_index()
        .rename(columns={'at_bat_number': 'min_ab'})
    )
    starters = ab_min[ab_min['min_ab'] == 1][['game_pk', 'pitcher']].copy()
    starters['season'] = season
    starter_gp.append(starters)
    print(f'[{season}] 선발 game_pk: {starters["game_pk"].nunique():,}경기')

starter_df = pd.concat(starter_gp, ignore_index=True)
all_game_pks = starter_df['game_pk'].unique().tolist()
print(f'\n전체 크롤링 대상: {len(all_game_pks):,}경기')

# ── game_pk 크롤링 (체크포인트 지원) ──────────────────────────────────────────
if os.path.exists(GP_CRAWLED_CSV):
    gp_done = pd.read_csv(GP_CRAWLED_CSV)
    crawled_gp = set(gp_done['game_pk'].tolist())
    play_records = gp_done.to_dict('records')
    print(f'체크포인트 로드: {len(crawled_gp):,}경기 완료')
else:
    crawled_gp   = set()
    play_records = []

remaining_gp = [gp for gp in all_game_pks if gp not in crawled_gp]
print(f'남은 크롤링: {len(remaining_gp):,}경기')

for i, game_pk in enumerate(remaining_gp):
    ids = crawl_play_ids_from_game(game_pk)
    season = starter_df[starter_df['game_pk'] == game_pk]['season'].iloc[0]
    for pid in ids:
        play_records.append({'play_id': pid, 'game_pk': game_pk, 'season': season})
    crawled_gp.add(game_pk)

    if (i + 1) % 50 == 0 or (i + 1) == len(remaining_gp):
        pd.DataFrame(play_records).to_csv(GP_CRAWLED_CSV, index=False)
        print(f'  [{i+1}/{len(remaining_gp)}] 체크포인트 저장 ({len(play_records):,}개 play_id)')

    time.sleep(0.5)

# ── X구간 필터 후 play_id CSV 저장 ───────────────────────────────────────────
raw_play_df = pd.DataFrame(play_records).drop_duplicates('play_id')

# 각 game_pk+pitcher별 최소 at_bat_number 기준으로 X구간 필터
# (game_pk 크롤링으로 수집하므로 at_bat 정보는 parquet에서 결합)
all_ab = []
for season in MY_SEASONS:
    parquet_path = os.path.join(PARQUET_DIR, f'statcast_{season}.parquet')
    if not os.path.exists(parquet_path):
        continue
    df = pd.read_parquet(parquet_path, columns=['game_pk', 'pitcher', 'at_bat_number', 'game_type'])
    df = df[df['game_type'] == 'R']
    all_ab.append(df)

ab_df = pd.concat(all_ab, ignore_index=True)
ab_min = (
    ab_df.groupby(['game_pk', 'pitcher'])['at_bat_number']
    .min().reset_index()
    .rename(columns={'at_bat_number': 'min_ab'})
)
ab_max_x = ab_min.copy()
ab_max_x['max_ab_x'] = ab_max_x['min_ab'] + X_MAX_BATTER - 1

# play_id에 at_bat_number 붙이기 위해 sv_id 대신 game_pk 기준으로 필터
# → game_pk 크롤링 결과의 play_id는 전체 경기 play_id이므로
#   X구간(처음 9타자) 에 해당하는 것만 남기기 위해 savant 개별 pitch 정보 필요
# → 단순화: 경기당 pitch 수 기준 상위 N% 제거보다
#   parquet의 at_bat_number 기준으로 play_id 유효 목록을 먼저 만들고
#   game_pk 매핑으로 play_id 결합

# X구간 at_bat 목록
ab_df2 = pd.concat(all_ab, ignore_index=True)
# at_bat_number 기준 선발 X구간 pitch 추출
ab_df2 = ab_df2.merge(ab_min, on=['game_pk', 'pitcher'])
ab_df2 = ab_df2[ab_df2['at_bat_number'] <= ab_df2['min_ab'] + X_MAX_BATTER - 1]
valid_gp = set(ab_df2['game_pk'].unique())

# game_pk 기준으로 X구간 경기만 남기기
play_df = raw_play_df[raw_play_df['game_pk'].isin(valid_gp)].copy()
play_df.to_csv(PLAY_ID_CSV, index=False)
print(f'\nplay_id 저장 완료: {len(play_df):,}개 → {PLAY_ID_CSV}')

## 2. 체크포인트 로드

In [ ]:
if os.path.exists(PROGRESS_CSV):
    progress = pd.read_csv(PROGRESS_CSV)
    done_ids = set(progress[progress['status'] == 'done']['play_id'].tolist())
    fail_ids = set(progress[progress['status'] == 'fail']['play_id'].tolist())
    print(f'체크포인트 로드: 완료 {len(done_ids):,}개 / 실패 {len(fail_ids):,}개')
else:
    progress = pd.DataFrame(columns=['play_id', 'season', 'batch_id', 'status'])
    done_ids = set()
    fail_ids = set()
    print('체크포인트 없음 → 처음부터 시작')

# play_id CSV 로드 (cell-4에서 저장했거나 기존 파일)
if not os.path.exists(PLAY_ID_CSV):
    raise FileNotFoundError(f'play_id CSV 없음: {PLAY_ID_CSV}\n먼저 셀 1(크롤링)을 실행하세요.')
play_df = pd.read_csv(PLAY_ID_CSV)

remaining = play_df[~play_df['play_id'].isin(done_ids)].reset_index(drop=True)
est_hours = len(remaining) * 5 / 3600
print(f'전체 play_id : {len(play_df):,}개')
print(f'남은 play_id : {len(remaining):,}개')
print(f'예상 배치 수 : {len(remaining) // BATCH_SIZE + 1}개')
print(f'예상 소요    : 약 {est_hours:.1f}시간')

## 3. 배치 다운로드 → zip → 드라이브 저장

In [ ]:
REQ_HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36'
}

def get_cdn_url(play_id):
    url  = f'https://baseballsavant.mlb.com/sporty-videos?playId={play_id}'
    res  = requests.get(url, headers=REQ_HEADERS, timeout=15)
    soup = BeautifulSoup(res.text, 'html.parser')
    video = soup.find('video')
    if video and video.find('source'):
        return video.find('source').get('src')
    return None

def download_mp4(cdn_url, save_path):
    with requests.get(cdn_url, headers=REQ_HEADERS, stream=True, timeout=30) as r:
        r.raise_for_status()
        with open(save_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

# 시즌별 zip 구분 (나 vs 팀원 겹치지 않게)
existing_zips = list(Path(VIDEO_DIR).glob(f'batch_{SEASON_KEY}_*.zip'))
start_batch   = len(existing_zips)
new_records   = []

for batch_idx, batch_start in enumerate(range(0, len(remaining), BATCH_SIZE)):
    batch_num  = start_batch + batch_idx + 1
    batch_name = f'batch_{SEASON_KEY}_{batch_num:04d}'
    zip_path   = os.path.join(VIDEO_DIR, f'{batch_name}.zip')

    if os.path.exists(zip_path):
        print(f'[{batch_name}] 이미 존재 → 건너뜀')
        continue

    batch_df  = remaining.iloc[batch_start:batch_start + BATCH_SIZE]
    batch_dir = os.path.join(TEMP_DIR, batch_name)
    os.makedirs(batch_dir, exist_ok=True)

    print(f'\n[{batch_name}] {len(batch_df)}개 다운로드 시작...')
    batch_done = 0
    batch_fail = 0

    for i, (_, row) in enumerate(batch_df.iterrows()):
        play_id  = row['play_id']
        season   = row['season']
        mp4_path = os.path.join(batch_dir, f'{play_id}.mp4')

        try:
            cdn_url = get_cdn_url(play_id)
            if cdn_url:
                download_mp4(cdn_url, mp4_path)
                new_records.append({'play_id': play_id, 'season': season,
                                    'batch_id': batch_name, 'status': 'done'})
                batch_done += 1
            else:
                new_records.append({'play_id': play_id, 'season': season,
                                    'batch_id': batch_name, 'status': 'fail'})
                batch_fail += 1
        except Exception:
            new_records.append({'play_id': play_id, 'season': season,
                                'batch_id': batch_name, 'status': 'fail'})
            batch_fail += 1

        if (i + 1) % 10 == 0:
            print(f'  {i+1}/{len(batch_df)} | 성공 {batch_done} / 실패 {batch_fail}')

        time.sleep(1.0)

    # zip 묶기
    print(f'[{batch_name}] zip 압축 중...')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for mp4 in Path(batch_dir).glob('*.mp4'):
            zf.write(mp4, mp4.name)

    zip_mb = os.path.getsize(zip_path) / 1024 / 1024
    print(f'[{batch_name}] 완료 ({zip_mb:.1f} MB) 성공 {batch_done} / 실패 {batch_fail}')

    shutil.rmtree(batch_dir)

    # 체크포인트 저장
    progress = pd.concat([progress, pd.DataFrame(new_records)], ignore_index=True)
    progress.to_csv(PROGRESS_CSV, index=False)
    new_records = []

print('\n전체 다운로드 완료!')

## 4. 현황 확인

In [ ]:
progress = pd.read_csv(PROGRESS_CSV)
print('=== 다운로드 현황 ===')
print(progress['status'].value_counts())
print(f'\n시즌별 현황:')
print(progress.groupby(['season', 'status']).size().unstack(fill_value=0))

zips = sorted(Path(VIDEO_DIR).glob(f'batch_{SEASON_KEY}_*.zip'))
total_gb = sum(z.stat().st_size for z in zips) / 1024**3
print(f'\n저장된 zip: {len(zips)}개 / 총 {total_gb:.2f} GB')